# 10 · KPI metric views

Governed **Unity Catalog metric views** (one per twin) over the `*_twin_current` tables. Each is a
`CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML` object that separates **dimensions** (slice-by)
from **measures** (aggregates queried via `MEASURE()`). One definition of each KPI, shared by the
AI/BI dashboard, Genie, SQL, and the agent. Requires DBR 17.2+ / serverless (YAML v1.1).

This notebook is the **source of truth** for the metric views — idempotent, re-run to rebuild.

## Config

In [ ]:
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
print("Metric views target:", FQ_MON)

## Definitions
Edit dimensions / measures here; the YAML is generated from these dicts.

In [ ]:
pass  # yaml import removed (unused on serverless); YAML built by hand below

def create_metric_view(name, source_table, comment, dimensions, measures):
    dims = "\n".join([f"  - name: {d[0]}\n    expr: {d[1]}" for d in dimensions])
    meas = "\n".join([f"  - name: {m[0]}\n    expr: {m[1]}" for m in measures])
    body = (
        f"version: 1.1\n"
        f"source: {FQ_MON}.{source_table}\n"
        f"comment: \"{comment}\"\n"
        f"dimensions:\n{dims}\n"
        f"measures:\n{meas}\n"
    )
    spark.sql(f"CREATE OR REPLACE VIEW {FQ_MON}.{name} WITH METRICS LANGUAGE YAML AS $$\n{body}$$")
    print("created", f"{FQ_MON}.{name}")

create_metric_view("demand_metrics", "demand_twin_current",
    "Governed demand KPIs over the demand twin (per material/plant/channel series).",
    [("Material","material"),("Material Group","material_group"),("Material Text","material_text"),
     ("Plant","plant"),("Channel","channel"),("Forecast Model","forecast_model"),("Top Customer Region","top_customer_region")],
    [("Series","COUNT(1)"),("Avg 13w Demand","AVG(demand_13w_avg)"),("Total Open Demand","SUM(open_demand_qty)"),
     ("Total Backlog","SUM(backlog_qty)"),("Total Horizon Forecast","SUM(horizon_total_forecast_qty)"),
     ("Avg WMAPE","AVG(forecast_wmape)"),("Avg Bias","AVG(forecast_bias)"),("Avg PI Coverage","AVG(forecast_pi_coverage)")])

create_metric_view("inventory_metrics", "inventory_twin_current",
    "Governed inventory KPIs over the inventory twin (per material/plant position).",
    [("Material","material"),("Material Group","material_group"),("Material Text","material_text"),("Plant","plant")],
    [("Positions","COUNT(1)"),("Stockouts","SUM(CASE WHEN is_stockout THEN 1 ELSE 0 END)"),
     ("Low Cover","SUM(CASE WHEN is_low_cover THEN 1 ELSE 0 END)"),("Overstock","SUM(CASE WHEN is_overstock THEN 1 ELSE 0 END)"),
     ("Total On Hand","SUM(on_hand_qty)"),("Avg Days Cover Demand","AVG(days_of_cover_demand)"),
     ("Avg Days Cover Actual","AVG(days_of_cover_actual)")])

create_metric_view("procurement_metrics", "procurement_twin_current",
    "Governed procurement KPIs over the procurement twin (per supplier/material/plant).",
    [("Supplier","supplier"),("Supplier Name","supplier_name"),("Supplier Country","supplier_country"),
     ("Material","material"),("Material Group","material_group"),("Plant","plant")],
    [("Supplier-Materials","COUNT(1)"),("Suppliers At Risk","SUM(CASE WHEN supplier_risk_flag THEN 1 ELSE 0 END)"),
     ("Avg OTIF","AVG(otif_rate)"),("Avg On-Time","AVG(on_time_rate)"),("Avg In-Full","AVG(in_full_rate)"),
     ("Avg Lead Time Days","AVG(avg_lead_time_days)"),("Total Open PO Qty","SUM(open_po_qty)"),("Avg Rejection Rate","AVG(rejection_rate)")])

create_metric_view("production_metrics", "production_twin_current",
    "Governed production KPIs over the production twin (per material/plant).",
    [("Material","material"),("Material Group","material_group"),("Material Text","material_text"),
     ("Plant","plant"),("Primary Work Centre","primary_work_centre")],
    [("Material-Plants","COUNT(1)"),("Bottlenecks","SUM(CASE WHEN is_bottleneck THEN 1 ELSE 0 END)"),
     ("Total Open Production Qty","SUM(open_production_qty)"),("Avg Yield Pct","AVG(yield_pct)"),
     ("Avg Scrap Pct","AVG(scrap_pct)"),("Avg First Pass Yield Pct","AVG(first_pass_yield_pct)"),
     ("Avg Schedule Adherence","AVG(schedule_adherence_rate)")])

create_metric_view("distribution_metrics", "distribution_twin_current",
    "Governed distribution KPIs over the distribution twin (per material/plant/channel).",
    [("Material","material"),("Material Group","material_group"),("Material Text","material_text"),
     ("Plant","plant"),("Channel","channel"),("Ship-To Region","top_ship_to_region")],
    [("Series","COUNT(1)"),("At Risk","SUM(CASE WHEN fulfillment_risk_flag THEN 1 ELSE 0 END)"),
     ("Avg Fill Rate","AVG(fill_rate)"),("Avg OTIF","AVG(otif_rate)"),("Avg On-Time","AVG(on_time_rate)"),
     ("Avg Delay Days","AVG(avg_delay_days)"),("Total Open Delivery Qty","SUM(open_delivery_qty)")])

## Verify
Measures must be wrapped in `MEASURE()`; `SELECT *` is not supported.

In [ ]:
display(spark.sql(f"""
SELECT `Material Group`, MEASURE(`Total Backlog`) AS backlog, MEASURE(`Avg WMAPE`) AS wmape
FROM {FQ_MON}.demand_metrics GROUP BY ALL ORDER BY backlog DESC
"""))
display(spark.sql(f"SELECT MEASURE(`Bottlenecks`) bottlenecks, MEASURE(`Avg Yield Pct`) yield_pct FROM {FQ_MON}.production_metrics"))